<a href="https://colab.research.google.com/github/NaavyaV/Accenture-1K-contract-review-challenge/blob/dev/Naavya_Vig_GPU_training_experiments_Capstone_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

!pip install -q "transformers>=4.45,<5" torch numpy pandas scikit-learn


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.30.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.0 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.


Import libraries and configure the model

In [2]:

import torch
import numpy as np

from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

MODEL_NAME = "distilbert/distilroberta-base"
NUM_LABELS = 41

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", DEVICE)


Using device: cuda


Initialize DistilRoBERTa

In [3]:

def initialize_model(
    model_name=MODEL_NAME,
    num_labels=NUM_LABELS
):
    tokenizer = AutoTokenizer.from_pretrained(
        model_name
    )

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=num_labels,
        problem_type="multi_label_classification"
    )

    model = model.to(DEVICE)

    return model, tokenizer


model, tokenizer = initialize_model()

print("Model initialized successfully!")
print("Number of labels:", model.config.num_labels)


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/331M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at distilbert/distilroberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model initialized successfully!
Number of labels: 41


Create the dataset class

In [4]:

class ContractDataset(Dataset):

    def __init__(self, texts, labels, tokenizer, max_length=256):
        assert len(texts) == len(labels)

        labels = np.asarray(labels, dtype=np.float32)

        assert labels.ndim == 2
        assert labels.shape[1] == NUM_LABELS

        self.encodings = tokenizer(
            texts,
            truncation=True,
            padding="max_length",
            max_length=max_length
        )

        self.labels = torch.tensor(
            labels,
            dtype=torch.float32
        )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):

        item = {
            key: torch.tensor(values[idx])
            for key, values in self.encodings.items()
        }

        item["labels"] = self.labels[idx]

        return item


Create reusable training function

In [5]:

def train_model(
    model,
    train_loader,
    epochs=3,
    learning_rate=2e-5
):
    optimizer = AdamW(
        model.parameters(),
        lr=learning_rate
    )

    for epoch in range(epochs):

        model.train()
        total_loss = 0

        for batch in train_loader:

            batch = {
                key: value.to(DEVICE)
                for key, value in batch.items()
            }

            optimizer.zero_grad()

            outputs = model(**batch)

            loss = outputs.loss

            loss.backward()

            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)

        print(
            f"Epoch {epoch + 1}/{epochs} "
            f"- Loss: {avg_loss:.4f}"
        )

    return model


Create reusable prediction function

In [6]:

def predict_clauses(
    text,
    model,
    tokenizer,
    threshold=0.5
):
    model.eval()

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=256
    ).to(DEVICE)

    with torch.no_grad():
        outputs = model(**inputs)

        probabilities = torch.sigmoid(
            outputs.logits
        )

    predictions = (
        probabilities >= threshold
    ).int()

    return {
        "probabilities": probabilities.cpu().numpy()[0],
        "predictions": predictions.cpu().numpy()[0]
    }


Create example training data

In [23]:

import pandas as pd
import numpy as np
from sklearn.preprocessing import MultiLabelBinarizer
from torch.utils.data import DataLoader

# 1. Load the three files uploaded individually to Colab
train_chunks = pd.read_json("/content/cuad_train_split.jsonl", lines=True)
val_chunks = pd.read_json("/content/cuad_val_split.jsonl", lines=True)
test_chunks = pd.read_json("/content/cuad_test_split.jsonl", lines=True)

# 2. Create a consistent mapping for all 41 categories
categories = sorted({
    category
    for labels in train_chunks["labels"]
    for category in labels
})

assert len(categories) == NUM_LABELS, (
    f"Expected {NUM_LABELS} categories, found {len(categories)}"
)

label_encoder = MultiLabelBinarizer(classes=categories)
label_encoder.fit([[]])

y_train = label_encoder.transform(train_chunks["labels"])
y_val = label_encoder.transform(val_chunks["labels"])
y_test = label_encoder.transform(test_chunks["labels"])

# 3. Build datasets using the ContractDataset class
train_dataset = ContractDataset(
    train_chunks["chunk_text"].fillna("").tolist(),
    y_train,
    tokenizer,
    max_length=512
)

val_dataset = ContractDataset(
    val_chunks["chunk_text"].fillna("").tolist(),
    y_val,
    tokenizer,
    max_length=512
)

test_dataset = ContractDataset(
    test_chunks["chunk_text"].fillna("").tolist(),
    y_test,
    tokenizer,
    max_length=512
)

# 4. Create data loaders
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)

# 5. Verify everything loaded
print("Training chunks:", len(train_dataset))
print("Validation chunks:", len(val_dataset))
print("Test chunks:", len(test_dataset))
print("Number of categories:", len(categories))
print("Label vector shape:", y_train.shape)

Training chunks: 11500
Validation chunks: 1998
Test chunks: 1226
Number of categories: 41
Label vector shape: (11500, 41)


In [35]:

import numpy as np
import pandas as pd
import re

splits = {
    "train": (train_chunks, np.asarray(y_train)),
    "validation": (val_chunks, np.asarray(y_val)),
    "test": (test_chunks, np.asarray(y_test))
}

# 1. Contract leakage
ids = {
    name: set(df["contract_id"].astype(str))
    for name, (df, _) in splits.items()
}

print("CONTRACT OVERLAP")
for a, b in [
    ("train", "validation"),
    ("train", "test"),
    ("validation", "test")
]:
    overlap = ids[a] & ids[b]
    print(f"{a} vs {b}: {len(overlap)} shared contracts")
    if overlap:
        print("Examples:", list(overlap)[:5])

# 2. Label distribution
rows = []

for j, category in enumerate(categories):
    row = {"category": category}
    for name, (_, labels) in splits.items():
        row[name + "_positives"] = int(labels[:, j].sum())
    rows.append(row)

freq = pd.DataFrame(rows).sort_values("train_positives")
display(freq)

# 3. Exact duplicate text across splits
def normalized_texts(df):
    return set(
        df["chunk_text"].fillna("")
        .astype(str)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    ) - {""}

texts = {
    name: normalized_texts(df)
    for name, (df, _) in splits.items()
}

print("\nEXACT TEXT OVERLAP")
for a, b in [
    ("train", "validation"),
    ("train", "test"),
    ("validation", "test")
]:
    print(a, b, len(texts[a] & texts[b]))

# 4. Positive examples for manual inspection
rare = freq.head(8)["category"].tolist()

for category in rare:
    j = list(categories).index(category)
    indices = np.flatnonzero(np.asarray(y_train)[:, j] == 1)

    print("\n" + "=" * 65)
    print(category, "| training positives:", len(indices))

    for i in indices[:2]:
        row = train_chunks.iloc[int(i)]
        print("Contract:", row["contract_id"])
        print("Labels:", row["labels"])
        print("TEXT:")
        print(str(row["chunk_text"])[:1200])
        print("-" * 50)

freq.to_csv("/content/cuad_label_audit.csv", index=False)
print("\nSaved: /content/cuad_label_audit.csv")


CONTRACT OVERLAP
train vs validation: 0 shared contracts
train vs test: 0 shared contracts
validation vs test: 0 shared contracts


,category,train_positives,validation_positives,test_positives
30,Price Restrictions,20,3,1
38,Unlimited/All-You-Can-Eat-License,22,2,1
34,Source Code Escrow,25,7,0
21,Most Favored Nation,25,6,4
22,No-Solicit Of Customers,33,8,8
36,Third Party Beneficiary,33,4,4
25,Non-Disparagement,38,8,7
1,Affiliate License-Licensor,39,3,11
23,No-Solicit Of Employees,56,7,7
17,Joint Ip Ownership,63,6,7



EXACT TEXT OVERLAP
train validation 1
train test 7
validation test 0

Price Restrictions | training positives: 20
Contract: LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT
Labels: ['Price Restrictions']
TEXT:
                       Section   2.4(a)  shall  be  subject  to   adjustment                            annually  on the  first  day  of  each  Product  Year                            beginning in the calendar  year 2000 and on the first                            day of each succeeding Product Year for the remainder                            of the Term and all  renewals  of this  Agreement  in                            proportion   to  the  increase  or  decrease  in  the                            Consumer  Price Index (CPI) as compared to the CPI as                            it  existed  on the  first  day of the  Term  of this                            Agreement.  The Company  also  reserves  the right to                            increase  or  decrease  the price 

Train the model

In [39]:

import os, gc, json, time, random
import numpy as np
import pandas as pd
import torch

from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import f1_score, precision_score, recall_score

# ==========================================
# CONFIGURATION
# ==========================================

MODEL_NAME = "microsoft/deberta-v3-base"

ROOT = "/content/cuad_deberta_v2"
BEST_PATH = ROOT + "/best_model"

MAX_LENGTH = 512
BATCH_SIZE = 4
GRAD_ACCUM = 2
EVAL_BATCH_SIZE = 16

TIME_LIMIT_MINUTES = 15
MAX_STEPS = 5000
EVAL_EVERY = 250

LEARNING_RATE = 1.5e-5
SEED = 42

os.makedirs(ROOT, exist_ok=True)

assert torch.cuda.is_available(), "Enable GPU in Colab"

DEVICE = torch.device("cuda")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ==========================================
# TOKENIZER + DATASETS
# ==========================================

new_tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

train_ds = ContractDataset(
    train_chunks["chunk_text"].fillna("").tolist(),
    y_train,
    new_tokenizer,
    max_length=MAX_LENGTH
)

val_ds = ContractDataset(
    val_chunks["chunk_text"].fillna("").tolist(),
    y_val,
    new_tokenizer,
    max_length=MAX_LENGTH
)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    pin_memory=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    pin_memory=True
)

# ==========================================
# MODEL
# ==========================================

gc.collect()
torch.cuda.empty_cache()

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(categories),
    problem_type="multi_label_classification"
).to(DEVICE)

# Moderate positive weighting
y = np.asarray(y_train)

positive_counts = y.sum(axis=0)
negative_counts = len(y) - positive_counts

weights = np.clip(
    np.sqrt(
        negative_counts / np.maximum(positive_counts, 1)
    ),
    1,
    3
)

loss_fn = torch.nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor(
        weights,
        dtype=torch.float32,
        device=DEVICE
    )
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=0.01
)

scaler = torch.amp.GradScaler("cuda")

# ==========================================
# EVALUATION
# ==========================================

@torch.no_grad()
def evaluate():
    model.eval()

    probabilities = []
    labels = []

    for batch in val_loader:
        inputs = {
            k: v.to(DEVICE)
            for k, v in batch.items()
            if k != "labels"
        }

        with torch.autocast("cuda", dtype=torch.float16):
            logits = model(**inputs).logits

        probabilities.append(
            torch.sigmoid(logits.float()).cpu().numpy()
        )
        labels.append(batch["labels"].numpy())

    probabilities = np.concatenate(probabilities)
    labels = np.concatenate(labels).astype(int)

    best = None

    for threshold in np.arange(0.10, 0.801, 0.025):
        predictions = (
            probabilities >= threshold
        ).astype(int)

        metrics = {
            "threshold": float(threshold),
            "micro_f1": float(f1_score(
                labels,
                predictions,
                average="micro",
                zero_division=0
            )),
            "macro_f1": float(f1_score(
                labels,
                predictions,
                average="macro",
                zero_division=0
            )),
            "precision": float(precision_score(
                labels,
                predictions,
                average="micro",
                zero_division=0
            )),
            "recall": float(recall_score(
                labels,
                predictions,
                average="micro",
                zero_division=0
            )),
            "categories_detected": int(
                (
                    f1_score(
                        labels,
                        predictions,
                        average=None,
                        zero_division=0
                    ) > 0
                ).sum()
            )
        }

        if best is None or (
            metrics["micro_f1"] > best["micro_f1"]
        ):
            best = metrics

    model.train()

    return best

# ==========================================
# TRAINING
# ==========================================

start_time = time.monotonic()
step = 0
best_micro = -1
history = []

model.train()
optimizer.zero_grad(set_to_none=True)

print("Starting DeBERTa training...")
print("GPU:", torch.cuda.get_device_name(0))

stop = False

while not stop:

    for batch in train_loader:

        inputs = {
            k: v.to(DEVICE)
            for k, v in batch.items()
            if k != "labels"
        }

        labels = batch["labels"].to(DEVICE).float()

        with torch.autocast("cuda", dtype=torch.float16):
            logits = model(**inputs).logits
            loss = loss_fn(
                logits.float(),
                labels
            ) / GRAD_ACCUM

        scaler.scale(loss).backward()

        step += 1

        if step % GRAD_ACCUM == 0:
            scaler.unscale_(optimizer)

            torch.nn.utils.clip_grad_norm_(
                model.parameters(), 1.0
            )

            scaler.step(optimizer)
            scaler.update()

            optimizer.zero_grad(set_to_none=True)

        if step % 50 == 0:
            elapsed = (
                time.monotonic() - start_time
            ) / 60

            print(
                f"Step {step} | "
                f"Loss {loss.item()*GRAD_ACCUM:.4f} | "
                f"Elapsed {elapsed:.1f} min"
            )

        if step % EVAL_EVERY == 0:

            metrics = evaluate()

            history.append({
                "step": step,
                **metrics
            })

            print("\nVALIDATION", step)
            print(metrics)

            if metrics["micro_f1"] > best_micro:

                best_micro = metrics["micro_f1"]

                model.save_pretrained(
                    BEST_PATH,
                    safe_serialization=True
                )

                new_tokenizer.save_pretrained(
                    BEST_PATH
                )

                with open(
                    BEST_PATH + "/metadata.json", "w"
                ) as f:
                    json.dump({
                        "step": step,
                        "metrics": metrics,
                        "categories": list(categories),
                        "max_length": MAX_LENGTH
                    }, f, indent=2)

                print("BEST CHECKPOINT SAVED")

        elapsed = (
            time.monotonic() - start_time
        ) / 60

        if (
            elapsed >= TIME_LIMIT_MINUTES
            or step >= MAX_STEPS
        ):
            stop = True
            break

# If time limit ended before first evaluation,
# still save a usable checkpoint.
if best_micro < 0:
    metrics = evaluate()
    best_micro = metrics["micro_f1"]
    model.save_pretrained(BEST_PATH)
    new_tokenizer.save_pretrained(BEST_PATH)
    with open(BEST_PATH + "/metadata.json", "w") as f:
        json.dump({
            "step": step,
            "metrics": metrics,
            "categories": list(categories),
            "max_length": MAX_LENGTH
        }, f, indent=2)

pd.DataFrame(history).to_csv(
    ROOT + "/history.csv",
    index=False
)

print("\nTRAINING COMPLETE")
print("Steps:", step)
print("Best validation micro-F1:", best_micro)
print("Saved at:", BEST_PATH)


tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/transformers/convert_slow_tokenizer.py:566: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/371M [00:00<?, ?B/s]

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Starting DeBERTa training...
GPU: Tesla T4
Step 50 | Loss 0.5122 | Elapsed 0.2 min
Step 100 | Loss 0.3698 | Elapsed 0.5 min
Step 150 | Loss 0.2323 | Elapsed 0.7 min
Step 200 | Loss 0.1446 | Elapsed 0.9 min
Step 250 | Loss 0.1838 | Elapsed 1.1 min

VALIDATION 250
{'threshold': 0.125, 'micro_f1': 0.05775216901781575, 'macro_f1': 0.01042852885618692, 'precision': 0.03130834410455263, 'recall': 0.37169650468883203, 'categories_detected': 7}
BEST CHECKPOINT SAVED
Step 300 | Loss 0.2220 | Elapsed 2.5 min
Step 350 | Loss 0.3154 | Elapsed 2.7 min
Step 400 | Loss 0.1029 | Elapsed 2.9 min
Step 450 | Loss 0.0584 | Elapsed 3.1 min
Step 500 | Loss 0.2367 | Elapsed 3.4 min

VALIDATION 500
{'threshold': 0.1, 'micro_f1': 0.07864726700747149, 'macro_f1': 0.0102400974087744, 'precision': 0.072992700729927, 'recall': 0.08525149190110827, 'categories_detected': 2}
BEST CHECKPOINT SAVED
Step 550 | Loss 0.4481 | Elapsed 4.8 min
Step 600 | Loss 0.0533 | Elapsed 5.0 min
Step 650 | Loss 0.0484 | Elapsed 5.2 mi

In [30]:

import os, gc, json, time, random, shutil
import numpy as np
import pandas as pd
import torch

from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import f1_score, precision_score, recall_score

# ============================================================
# 1. CONFIGURATION
# ============================================================

SEED = 42
MODEL_NAME = "distilbert/distilroberta-base"
MAX_LENGTH = 512
BATCH_SIZE = 8

# 10-minute approximate budget
TRAIN_TIME_LIMIT = 525
EVAL_EVERY = 400
MAX_STEPS = 5000

# Learning rate progression based on your successful runs
LR_PHASES = [
    (1800, 2e-5),
    (3000, 1e-5),
    (5000, 5e-6)
]

WEIGHT_CAP = 3.0

ROOT = "/content/cuad_fresh_training"
BEST_MICRO = ROOT + "/best_micro"
BEST_MACRO = ROOT + "/best_macro"
HISTORY_PATH = ROOT + "/history.csv"

os.makedirs(ROOT, exist_ok=True)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
assert DEVICE.type == "cuda", "Enable GPU in Colab."

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

start_time = time.monotonic()

# ============================================================
# 2. REBUILD DATASETS AT 512 TOKENS
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_dataset = ContractDataset(
    train_chunks["chunk_text"].fillna("").tolist(),
    y_train,
    tokenizer,
    max_length=MAX_LENGTH
)

val_dataset = ContractDataset(
    val_chunks["chunk_text"].fillna("").tolist(),
    y_val,
    tokenizer,
    max_length=MAX_LENGTH
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    pin_memory=True
)

print("Training examples:", len(train_dataset))
print("Validation examples:", len(val_dataset))
print("Max token length:", MAX_LENGTH)

# ============================================================
# 3. FRESH MODEL
# ============================================================

if "model" in globals():
    del model

gc.collect()
torch.cuda.empty_cache()

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(categories),
    problem_type="multi_label_classification"
).to(DEVICE)

# ============================================================
# 4. CLASS-WEIGHTED LOSS
# ============================================================

y_train_np = np.asarray(y_train, dtype=np.float32)

positives = y_train_np.sum(axis=0)
negatives = len(y_train_np) - positives

class_weights = np.clip(
    np.sqrt(negatives / np.maximum(positives, 1)),
    1.0,
    WEIGHT_CAP
)

criterion = torch.nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor(
        class_weights,
        dtype=torch.float32,
        device=DEVICE
    )
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=2e-5,
    weight_decay=0.01
)

scaler = torch.amp.GradScaler("cuda")

# ============================================================
# 5. EVALUATION
# ============================================================

@torch.no_grad()
def evaluate_model(model):
    model.eval()

    all_probs = []
    all_true = []

    for batch in val_loader:
        inputs = {
            k: v.to(DEVICE, non_blocking=True)
            for k, v in batch.items()
            if k != "labels"
        }

        with torch.autocast("cuda", dtype=torch.float16):
            logits = model(**inputs).logits

        all_probs.append(
            torch.sigmoid(logits.float()).cpu().numpy()
        )
        all_true.append(batch["labels"].numpy())

    probs = np.concatenate(all_probs)
    true = np.concatenate(all_true).astype(int)

    # One global threshold: avoids rare-label overprediction
    best = None

    for threshold in np.arange(0.10, 0.801, 0.025):
        pred = (probs >= threshold).astype(int)

        micro = f1_score(
            true, pred,
            average="micro",
            zero_division=0
        )

        macro = f1_score(
            true, pred,
            average="macro",
            zero_division=0
        )

        if best is None or micro > best["micro_f1"]:
            per_class = f1_score(
                true, pred,
                average=None,
                zero_division=0
            )

            best = {
                "threshold": float(threshold),
                "micro_f1": float(micro),
                "macro_f1": float(macro),
                "precision": float(precision_score(
                    true, pred,
                    average="micro",
                    zero_division=0
                )),
                "recall": float(recall_score(
                    true, pred,
                    average="micro",
                    zero_division=0
                )),
                "categories_detected": int(
                    (per_class > 0).sum()
                ),
                "predicted_positives": int(pred.sum())
            }

    # Separately find the threshold that maximizes macro-F1,
    # subject to minimum precision and micro-F1 requirements.
    macro_candidate = None

    for threshold in np.arange(0.10, 0.801, 0.025):
        pred = (probs >= threshold).astype(int)

        micro = f1_score(
            true, pred,
            average="micro",
            zero_division=0
        )
        macro = f1_score(
            true, pred,
            average="macro",
            zero_division=0
        )
        precision = precision_score(
            true, pred,
            average="micro",
            zero_division=0
        )

        if precision < 0.35 or micro < 0.40:
            continue

        if (
            macro_candidate is None
            or macro > macro_candidate["macro_f1"]
        ):
            per_class = f1_score(
                true, pred,
                average=None,
                zero_division=0
            )

            macro_candidate = {
                "threshold": float(threshold),
                "micro_f1": float(micro),
                "macro_f1": float(macro),
                "precision": float(precision),
                "recall": float(recall_score(
                    true, pred,
                    average="micro",
                    zero_division=0
                )),
                "categories_detected": int(
                    (per_class > 0).sum()
                ),
                "predicted_positives": int(pred.sum())
            }

    model.train()
    return best, macro_candidate

# ============================================================
# 6. CHECKPOINT SAVING
# ============================================================

def save_checkpoint(path, metrics, step, objective):
    os.makedirs(path, exist_ok=True)

    model.save_pretrained(path)
    tokenizer.save_pretrained(path)

    with open(path + "/metadata.json", "w") as f:
        json.dump({
            "step": step,
            "objective": objective,
            "metrics": metrics,
            "categories": list(categories),
            "max_length": MAX_LENGTH,
            "weight_cap": WEIGHT_CAP,
            "seed": SEED
        }, f, indent=2)

# ============================================================
# 7. TRAINING
# ============================================================

best_micro_f1 = -1
best_macro_f1 = -1

history = []
step = 0
losses = []

model.train()

def learning_rate_at(step):
    for boundary, lr in LR_PHASES:
        if step < boundary:
            return lr
    return LR_PHASES[-1][1]

def run_validation(step):
    global best_micro_f1, best_macro_f1

    micro_metrics, macro_metrics = evaluate_model(model)

    row = {
        "step": step,
        "lr": learning_rate_at(step - 1),
        **micro_metrics,
        "macro_optimized_f1": (
            macro_metrics["macro_f1"]
            if macro_metrics is not None else np.nan
        )
    }
    history.append(row)

    print(
        f"\nVALIDATION @ {step} | "
        f"Micro={micro_metrics['micro_f1']:.4f} | "
        f"Macro={micro_metrics['macro_f1']:.4f} | "
        f"Precision={micro_metrics['precision']:.4f} | "
        f"Recall={micro_metrics['recall']:.4f} | "
        f"Categories={micro_metrics['categories_detected']}"
    )

    if micro_metrics["micro_f1"] > best_micro_f1:
        best_micro_f1 = micro_metrics["micro_f1"]
        save_checkpoint(
            BEST_MICRO, micro_metrics, step, "micro"
        )
        print("NEW BEST MICRO MODEL SAVED")

    if (
        macro_metrics is not None
        and macro_metrics["macro_f1"] > best_macro_f1
    ):
        best_macro_f1 = macro_metrics["macro_f1"]
        save_checkpoint(
            BEST_MACRO, macro_metrics, step, "macro"
        )
        print("NEW BEST MACRO MODEL SAVED")

    pd.DataFrame(history).to_csv(
        HISTORY_PATH, index=False
    )

while step < MAX_STEPS:

    for batch in train_loader:

        if step >= MAX_STEPS:
            break

        if time.monotonic() - start_time >= TRAIN_TIME_LIMIT:
            break

        lr = learning_rate_at(step)

        for group in optimizer.param_groups:
            group["lr"] = lr

        inputs = {
            k: v.to(DEVICE, non_blocking=True)
            for k, v in batch.items()
            if k != "labels"
        }

        labels = batch["labels"].to(
            DEVICE, non_blocking=True
        ).float()

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast("cuda", dtype=torch.float16):
            logits = model(**inputs).logits
            loss = criterion(logits.float(), labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)

        torch.nn.utils.clip_grad_norm_(
            model.parameters(), 1.0
        )

        scaler.step(optimizer)
        scaler.update()

        step += 1
        losses.append(loss.item())

        if step % 100 == 0:
            print(
                f"Step {step}/{MAX_STEPS} | "
                f"LR {lr:.1e} | "
                f"Loss {np.mean(losses[-100:]):.4f} | "
                f"Elapsed "
                f"{(time.monotonic()-start_time)/60:.2f} min"
            )

        if step % EVAL_EVERY == 0:
            run_validation(step)

    if time.monotonic() - start_time >= TRAIN_TIME_LIMIT:
        break

# Final evaluation for unfinished interval
if step > 0 and step % EVAL_EVERY != 0:
    run_validation(step)

# ============================================================
# 8. FINAL RESULTS
# ============================================================

results_df = pd.DataFrame(history)

print("\n" + "=" * 60)
print("FINAL TRAINING RESULTS")
print("=" * 60)

display(results_df)

print("Best micro-F1:", round(best_micro_f1, 4))
print("Best macro-F1:", round(best_macro_f1, 4))
print("Steps completed:", step)
print(
    "Runtime:",
    round((time.monotonic()-start_time)/60, 2),
    "minutes"
)

print("\nBest overall model:", BEST_MICRO)
print("Best category-balanced model:", BEST_MACRO)
print("Training history:", HISTORY_PATH)


Training examples: 11500
Validation examples: 1998
Max token length: 512


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at distilbert/distilroberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Step 100/5000 | LR 2.0e-05 | Loss 0.3624 | Elapsed 0.44 min
Step 200/5000 | LR 2.0e-05 | Loss 0.2039 | Elapsed 0.65 min
Step 300/5000 | LR 2.0e-05 | Loss 0.1810 | Elapsed 0.85 min
Step 400/5000 | LR 2.0e-05 | Loss 0.1796 | Elapsed 1.06 min

VALIDATION @ 400 | Micro=0.2123 | Macro=0.0475 | Precision=0.3019 | Recall=0.1637 | Categories=5
NEW BEST MICRO MODEL SAVED
Step 500/5000 | LR 2.0e-05 | Loss 0.1813 | Elapsed 1.58 min
Step 600/5000 | LR 2.0e-05 | Loss 0.1637 | Elapsed 1.79 min
Step 700/5000 | LR 2.0e-05 | Loss 0.1699 | Elapsed 2.00 min
Step 800/5000 | LR 2.0e-05 | Loss 0.1728 | Elapsed 2.22 min

VALIDATION @ 800 | Micro=0.2440 | Macro=0.0607 | Precision=0.4652 | Recall=0.1654 | Categories=5
NEW BEST MICRO MODEL SAVED
Step 900/5000 | LR 2.0e-05 | Loss 0.1558 | Elapsed 2.75 min
Step 1000/5000 | LR 2.0e-05 | Loss 0.1638 | Elapsed 2.96 min
Step 1100/5000 | LR 2.0e-05 | Loss 0.1441 | Elapsed 3.18 min
Step 1200/5000 | LR 2.0e-05 | Loss 0.1420 | Elapsed 3.40 min

VALIDATION @ 1200 | Micro=

,step,lr,threshold,micro_f1,macro_f1,precision,recall,categories_detected,predicted_positives,macro_optimized_f1
0,400,0.00002,0.225,0.212272,0.047481,0.301887,0.163683,5,636,NaN
1,800,0.00002,0.225,0.244025,0.060685,0.465228,0.165388,5,417,NaN
2,1200,0.00002,0.350,0.369835,0.111625,0.469201,0.305200,9,763,NaN
3,1600,0.00002,0.475,0.482725,0.201229,0.562358,0.422847,14,882,0.217928
4,2000,0.00001,0.425,0.507856,0.265784,0.505922,0.509804,22,1182,0.277652
5,2400,0.00001,0.550,0.544439,0.260830,0.599385,0.498721,19,976,0.285966
6,2800,0.00001,0.475,0.561308,0.276646,0.600583,0.526854,19,1029,0.331821


Best micro-F1: 0.5613
Best macro-F1: 0.3318
Steps completed: 2800
Runtime: 8.81 minutes

Best overall model: /content/cuad_fresh_training/best_micro
Best category-balanced model: /content/cuad_fresh_training/best_macro
Training history: /content/cuad_fresh_training/history.csv


In [32]:

import os, json
import numpy as np
import pandas as pd
import torch

from torch.utils.data import DataLoader
from transformers import AutoModelForSequenceClassification
from sklearn.metrics import f1_score, precision_score, recall_score

MODEL_PATH = "/content/cuad_fresh_training/best_micro"
OUTPUT_PATH = "/content/cuad_fresh_training/calibration"
os.makedirs(OUTPUT_PATH, exist_ok=True)

with open(MODEL_PATH + "/metadata.json") as f:
    metadata = json.load(f)

base_threshold = float(metadata["metrics"]["threshold"])

m = AutoModelForSequenceClassification.from_pretrained(
    MODEL_PATH
).to(DEVICE)
m.eval()

loader = DataLoader(
    val_dataset, batch_size=32, shuffle=False
)

probs_list = []
true_list = []

with torch.no_grad():
    for batch in loader:
        inputs = {
            k: v.to(DEVICE)
            for k, v in batch.items()
            if k != "labels"
        }

        with torch.autocast("cuda", dtype=torch.float16):
            logits = m(**inputs).logits

        probs_list.append(
            torch.sigmoid(logits.float()).cpu().numpy()
        )
        true_list.append(batch["labels"].numpy())

probs = np.concatenate(probs_list)
true = np.concatenate(true_list).astype(int)

del m
torch.cuda.empty_cache()

def score(thresholds):
    pred = (probs >= thresholds).astype(int)

    return {
        "micro_f1": f1_score(
            true, pred, average="micro", zero_division=0
        ),
        "macro_f1": f1_score(
            true, pred, average="macro", zero_division=0
        ),
        "precision": precision_score(
            true, pred, average="micro", zero_division=0
        ),
        "recall": recall_score(
            true, pred, average="micro", zero_division=0
        ),
        "categories_detected": int(
            (f1_score(
                true, pred, average=None, zero_division=0
            ) > 0).sum()
        ),
        "predicted_positives": int(pred.sum())
    }

# Coordinate-descent optimization:
# optimize the combined objective rather than each class alone.
#
# Constraints:
# - micro-F1 cannot fall more than 0.02 below baseline
# - total predictions cannot exceed 1.5x baseline
# - minimum 10 validation positives before class-specific tuning

base = np.full(len(categories), base_threshold)
base_metrics = score(base)

thresholds = base.copy()

def objective(metrics):
    if metrics["micro_f1"] < base_metrics["micro_f1"] - 0.02:
        return -1e9

    if metrics["predicted_positives"] > (
        base_metrics["predicted_positives"] * 1.5
    ):
        return -1e9

    return (
        0.65 * metrics["micro_f1"]
        + 0.35 * metrics["macro_f1"]
    )

candidates = np.arange(0.15, 0.751, 0.025)

for iteration in range(3):
    for j in range(len(categories)):

        if true[:, j].sum() < 10:
            continue

        current_best = objective(score(thresholds))
        best_t = thresholds[j]

        for t in candidates:
            trial = thresholds.copy()
            trial[j] = t

            value = objective(score(trial))

            if value > current_best + 1e-8:
                current_best = value
                best_t = t

        thresholds[j] = best_t

    print(
        f"Iteration {iteration+1}:",
        score(thresholds)
    )

final_metrics = score(thresholds)

print("\nBASELINE:", base_metrics)
print("CALIBRATED:", final_metrics)

threshold_table = pd.DataFrame({
    "category": categories,
    "threshold": thresholds,
    "validation_positives": true.sum(axis=0)
})

display(threshold_table)

with open(OUTPUT_PATH + "/thresholds.json", "w") as f:
    json.dump({
        "model_path": MODEL_PATH,
        "thresholds": thresholds.tolist(),
        "metrics": final_metrics
    }, f, indent=2)

threshold_table.to_csv(
    OUTPUT_PATH + "/thresholds.csv",
    index=False
)

print("\nSaved calibration to:", OUTPUT_PATH)


Iteration 1: {'micro_f1': 0.63003663003663, 'macro_f1': 0.3985169808084398, 'precision': 0.6805143422354105, 'recall': 0.5865302642796248, 'categories_detected': 28, 'predicted_positives': 1011}
Iteration 2: {'micro_f1': 0.6308539944903582, 'macro_f1': 0.39716196725830427, 'precision': 0.6835820895522388, 'recall': 0.5856777493606138, 'categories_detected': 28, 'predicted_positives': 1005}
Iteration 3: {'micro_f1': 0.6308539944903582, 'macro_f1': 0.39716196725830427, 'precision': 0.6835820895522388, 'recall': 0.5856777493606138, 'categories_detected': 28, 'predicted_positives': 1005}

BASELINE: {'micro_f1': 0.5613079019073569, 'macro_f1': 0.2766462519330046, 'precision': 0.6005830903790087, 'recall': 0.5268542199488491, 'categories_detected': 19, 'predicted_positives': 1029}
CALIBRATED: {'micro_f1': 0.6308539944903582, 'macro_f1': 0.39716196725830427, 'precision': 0.6835820895522388, 'recall': 0.5856777493606138, 'categories_detected': 28, 'predicted_positives': 1005}


,category,threshold,validation_positives
0,Affiliate License-Licensee,0.275,18
1,Affiliate License-Licensor,0.475,3
2,Agreement Date,0.700,55
3,Anti-Assignment,0.450,67
4,Audit Rights,0.450,71
5,Cap On Liability,0.275,57
6,Change Of Control,0.250,29
7,Competitive Restriction Exception,0.475,10
8,Covenant Not To Sue,0.250,17
9,Document Name,0.650,57



Saved calibration to: /content/cuad_fresh_training/calibration


In [33]:

import os
import json
import shutil
from google.colab import files

ROOT = "/content/cuad_fresh_training"
MODEL_PATH = f"{ROOT}/best_micro"
CALIBRATION_PATH = f"{ROOT}/calibration/thresholds.json"

EXPORT_DIR = "/content/CUAD_Classifier_v1"
ZIP_PATH = "/content/CUAD_Classifier_v1.zip"

# Verify required files
assert os.path.isdir(MODEL_PATH), "Model folder not found"
assert os.path.isfile(CALIBRATION_PATH), "Calibration file not found"

# Recreate clean export folder
if os.path.exists(EXPORT_DIR):
    shutil.rmtree(EXPORT_DIR)

os.makedirs(EXPORT_DIR)

# Copy trained model and tokenizer
shutil.copytree(
    MODEL_PATH,
    f"{EXPORT_DIR}/model"
)

# Copy calibrated thresholds
shutil.copy2(
    CALIBRATION_PATH,
    f"{EXPORT_DIR}/thresholds.json"
)

# Copy training history
history_path = f"{ROOT}/history.csv"

if os.path.exists(history_path):
    shutil.copy2(
        history_path,
        f"{EXPORT_DIR}/training_history.csv"
    )

# Create model information
with open(CALIBRATION_PATH) as f:
    calibration = json.load(f)

with open(f"{MODEL_PATH}/metadata.json") as f:
    metadata = json.load(f)

model_info = {
    "model_name": "CUAD Multi-Label Contract Classifier",
    "version": "1.0",
    "base_model": "distilbert/distilroberta-base",
    "num_categories": len(categories),
    "max_length": 512,
    "categories": list(categories),
    "validation_metrics": calibration["metrics"],
    "calibration_method": "Per-category thresholds",
    "training_checkpoint": metadata,
    "notes": (
        "Metrics are from validation data used for "
        "threshold calibration, not independent test data."
    )
}

with open(f"{EXPORT_DIR}/model_info.json", "w") as f:
    json.dump(model_info, f, indent=2)

# Zip everything
shutil.make_archive(
    "/content/CUAD_Classifier_v1",
    "zip",
    EXPORT_DIR
)

size_mb = os.path.getsize(ZIP_PATH) / (1024 ** 2)

print("MODEL EXPORTED SUCCESSFULLY")
print(f"ZIP size: {size_mb:.1f} MB")
print("Location:", ZIP_PATH)
print("Micro-F1:", calibration["metrics"]["micro_f1"])
print("Macro-F1:", calibration["metrics"]["macro_f1"])

# Download to your computer
files.download(ZIP_PATH)


MODEL EXPORTED SUCCESSFULLY
ZIP size: 292.3 MB
Location: /content/CUAD_Classifier_v1.zip
Micro-F1: 0.6308539944903582
Macro-F1: 0.39716196725830427


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [34]:

import json
import numpy as np
import torch

from torch.utils.data import DataLoader
from transformers import AutoModelForSequenceClassification
from sklearn.metrics import (
    f1_score, precision_score, recall_score
)

EXPORT_DIR = "/content/CUAD_Classifier_v1"

with open(f"{EXPORT_DIR}/thresholds.json") as f:
    thresholds = np.array(
        json.load(f)["thresholds"]
    )

with open(f"{EXPORT_DIR}/model_info.json") as f:
    info = json.load(f)

assert list(categories) == info["categories"], (
    "Category order does not match the saved model"
)

# Rebuild test data at 512 tokens
test_dataset_512 = ContractDataset(
    test_chunks["chunk_text"].fillna("").tolist(),
    y_test,
    tokenizer,
    max_length=512
)

test_loader = DataLoader(
    test_dataset_512,
    batch_size=32,
    shuffle=False
)

test_model = AutoModelForSequenceClassification.from_pretrained(
    f"{EXPORT_DIR}/model"
).to(DEVICE)

test_model.eval()

all_probs = []
all_labels = []

with torch.no_grad():
    for batch in test_loader:
        inputs = {
            k: v.to(DEVICE)
            for k, v in batch.items()
            if k != "labels"
        }

        with torch.autocast("cuda", dtype=torch.float16):
            logits = test_model(**inputs).logits

        all_probs.append(
            torch.sigmoid(logits.float()).cpu().numpy()
        )
        all_labels.append(batch["labels"].numpy())

probs = np.concatenate(all_probs)
true = np.concatenate(all_labels).astype(int)

assert probs.shape[1] == len(thresholds)

pred = (probs >= thresholds[None, :]).astype(int)

print("FINAL TEST RESULTS")
print("=" * 45)

print("Micro-F1:", f1_score(
    true, pred, average="micro", zero_division=0
))
print("Macro-F1:", f1_score(
    true, pred, average="macro", zero_division=0
))
print("Precision:", precision_score(
    true, pred, average="micro", zero_division=0
))
print("Recall:", recall_score(
    true, pred, average="micro", zero_division=0
))

per_class = f1_score(
    true, pred, average=None, zero_division=0
)

print(
    "Categories detected:",
    int((per_class > 0).sum()),
    "/",
    len(categories)
)


FINAL TEST RESULTS
Micro-F1: 0.6187389510901591
Macro-F1: 0.3636704898777126
Precision: 0.6595477386934674
Recall: 0.5826859045504994
Categories detected: 26 / 41


In [31]:

import json
import numpy as np
import pandas as pd
import torch

from torch.utils.data import DataLoader
from transformers import AutoModelForSequenceClassification
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    average_precision_score
)

paths = {
    "best_micro": "/content/cuad_fresh_training/best_micro",
    "best_macro": "/content/cuad_fresh_training/best_macro"
}

loader = DataLoader(
    val_dataset, batch_size=32, shuffle=False
)

@torch.no_grad()
def get_predictions(path):
    m = AutoModelForSequenceClassification.from_pretrained(
        path
    ).to(DEVICE)
    m.eval()

    probs = []
    true = []

    for batch in loader:
        inputs = {
            k: v.to(DEVICE)
            for k, v in batch.items()
            if k != "labels"
        }

        with torch.autocast("cuda", dtype=torch.float16):
            logits = m(**inputs).logits

        probs.append(
            torch.sigmoid(logits.float()).cpu().numpy()
        )
        true.append(batch["labels"].numpy())

    del m
    torch.cuda.empty_cache()

    return np.concatenate(probs), np.concatenate(true).astype(int)

for name, path in paths.items():
    with open(path + "/metadata.json") as f:
        metadata = json.load(f)

    threshold = metadata["metrics"]["threshold"]

    probs, true = get_predictions(path)
    pred = (probs >= threshold).astype(int)

    rows = []

    for j, category in enumerate(categories):
        positives = int(true[:, j].sum())

        ap = (
            average_precision_score(true[:, j], probs[:, j])
            if positives > 0 else np.nan
        )

        rows.append({
            "category": category,
            "actual": positives,
            "predicted": int(pred[:, j].sum()),
            "precision": precision_score(
                true[:, j], pred[:, j], zero_division=0
            ),
            "recall": recall_score(
                true[:, j], pred[:, j], zero_division=0
            ),
            "f1": f1_score(
                true[:, j], pred[:, j], zero_division=0
            ),
            "average_precision": ap,
            "mean_positive_probability": (
                float(probs[true[:, j] == 1, j].mean())
                if positives > 0 else np.nan
            ),
            "mean_negative_probability": (
                float(probs[true[:, j] == 0, j].mean())
                if (true[:, j] == 0).any() else np.nan
            )
        })

    df = pd.DataFrame(rows).sort_values(
        ["f1", "average_precision"],
        ascending=[True, False]
    )

    print(f"\n{name.upper()} — threshold {threshold:.3f}")
    print("Micro-F1:", round(
        f1_score(true, pred, average="micro"), 4
    ))
    print("Macro-F1:", round(
        f1_score(true, pred, average="macro"), 4
    ))

    display(df)

    df.to_csv(
        f"/content/cuad_{name}_category_diagnostics.csv",
        index=False
    )



BEST_MICRO — threshold 0.475
Micro-F1: 0.5613
Macro-F1: 0.2766


,category,actual,predicted,precision,recall,f1,average_precision,mean_positive_probability,mean_negative_probability
15,Ip Ownership Assignment,28,0,0.000000,0.000000,0.000000,0.659457,0.293692,0.038500
16,Irrevocable Or Perpetual License,23,0,0.000000,0.000000,0.000000,0.643158,0.306609,0.018183
0,Affiliate License-Licensee,18,0,0.000000,0.000000,0.000000,0.513987,0.256148,0.018617
37,Uncapped Liability,18,2,0.000000,0.000000,0.000000,0.469043,0.233994,0.016777
39,Volume Restriction,18,0,0.000000,0.000000,0.000000,0.370887,0.163719,0.032632
40,Warranty Duration,19,0,0.000000,0.000000,0.000000,0.288741,0.101592,0.026200
24,Non-Compete,14,0,0.000000,0.000000,0.000000,0.268628,0.188943,0.051598
8,Covenant Not To Sue,17,0,0.000000,0.000000,0.000000,0.178771,0.132547,0.032754
6,Change Of Control,29,0,0.000000,0.000000,0.000000,0.145980,0.130803,0.035874
33,Rofr/Rofo/Rofn,19,0,0.000000,0.000000,0.000000,0.095600,0.177434,0.050199



BEST_MACRO — threshold 0.275
Micro-F1: 0.5258
Macro-F1: 0.3318


,category,actual,predicted,precision,recall,f1,average_precision,mean_positive_probability,mean_negative_probability
39,Volume Restriction,18,0,0.000000,0.000000,0.000000,0.370887,0.163719,0.032632
8,Covenant Not To Sue,17,0,0.000000,0.000000,0.000000,0.178771,0.132547,0.032754
1,Affiliate License-Licensor,3,0,0.000000,0.000000,0.000000,0.094435,0.111540,0.012645
23,No-Solicit Of Employees,7,0,0.000000,0.000000,0.000000,0.091660,0.069873,0.018115
7,Competitive Restriction Exception,10,0,0.000000,0.000000,0.000000,0.086362,0.104541,0.025236
19,Liquidated Damages,4,0,0.000000,0.000000,0.000000,0.057131,0.082867,0.025491
21,Most Favored Nation,6,0,0.000000,0.000000,0.000000,0.031959,0.033165,0.010850
22,No-Solicit Of Customers,8,0,0.000000,0.000000,0.000000,0.026485,0.028204,0.011371
34,Source Code Escrow,7,0,0.000000,0.000000,0.000000,0.017461,0.022283,0.009679
25,Non-Disparagement,8,0,0.000000,0.000000,0.000000,0.012248,0.020440,0.013972


In [25]:

import json
import numpy as np
import pandas as pd
import torch

from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer
)
from sklearn.metrics import (
    precision_recall_fscore_support
)

BEST_PATH = "/content/cuad_best_model"

model = AutoModelForSequenceClassification.from_pretrained(
    BEST_PATH
).to(DEVICE)

tokenizer = AutoTokenizer.from_pretrained(BEST_PATH)

with open(f"{BEST_PATH}/training_config.json") as f:
    saved_config = json.load(f)

threshold = saved_config["best_threshold"]

probs, true = get_validation_outputs(
    model, validation_loader
)

print("BEST MODEL RESULTS")

for t in [0.5, threshold]:
    metrics = calculate_metrics(probs, true, t)
    print(f"\nThreshold: {t:.3f}")
    print(metrics)

predictions = (probs >= threshold).astype(int)

p, r, f1, support = precision_recall_fscore_support(
    true,
    predictions,
    average=None,
    zero_division=0
)

per_category = pd.DataFrame({
    "category": categories,
    "precision": p,
    "recall": r,
    "f1": f1,
    "actual_count": support,
    "predicted_count": predictions.sum(axis=0)
})

print("\nBEST PERFORMING CATEGORIES")
display(
    per_category.sort_values(
        "f1", ascending=False
    ).head(15)
)

print("\nWORST PERFORMING CATEGORIES")
display(
    per_category.sort_values(
        "f1", ascending=True
    ).head(15)
)

per_category.to_csv(
    "/content/cuad_transformer_category_metrics.csv",
    index=False
)


BEST MODEL RESULTS

Threshold: 0.500
{'micro_f1': 0.25357873210633947, 'macro_f1': 0.06659059514781399, 'precision': 0.6326530612244898, 'recall': 0.1585677749360614, 'predicted_positives': 294}

Threshold: 0.260
{'micro_f1': 0.2754237288135593, 'macro_f1': 0.08904916259368956, 'precision': 0.36363636363636365, 'recall': 0.2216538789428815, 'predicted_positives': 715}

BEST PERFORMING CATEGORIES


,category,precision,recall,f1,actual_count,predicted_count
28,Parties,0.747126,0.764706,0.755814,85,87
2,Agreement Date,0.580247,0.854545,0.691176,55,81
9,Document Name,0.560976,0.807018,0.661871,57,82
10,Effective Date,0.419753,0.723404,0.531250,47,81
12,Expiration Date,0.306122,0.263158,0.283019,57,49
13,Governing Law,0.176871,0.509804,0.262626,51,147
29,Post-Termination Services,0.140351,0.195122,0.163265,41,57
3,Anti-Assignment,0.180000,0.134328,0.153846,67,50
5,Cap On Liability,0.128205,0.175439,0.148148,57,78
8,Covenant Not To Sue,0.000000,0.000000,0.000000,17,0



WORST PERFORMING CATEGORIES


,category,precision,recall,f1,actual_count,predicted_count
0,Affiliate License-Licensee,0.0,0.0,0.0,18,0
1,Affiliate License-Licensor,0.0,0.0,0.0,3,0
6,Change Of Control,0.0,0.0,0.0,29,0
4,Audit Rights,0.0,0.0,0.0,71,3
7,Competitive Restriction Exception,0.0,0.0,0.0,10,0
14,Insurance,0.0,0.0,0.0,31,0
11,Exclusivity,0.0,0.0,0.0,45,0
8,Covenant Not To Sue,0.0,0.0,0.0,17,0
15,Ip Ownership Assignment,0.0,0.0,0.0,28,0
22,No-Solicit Of Customers,0.0,0.0,0.0,8,0


In [26]:

import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, precision_score, recall_score

# Assumes 'probs' and 'true' are from the full validation set
assert probs.shape == true.shape
assert probs.shape[1] == len(categories)

global_threshold = 0.26
thresholds = np.full(len(categories), global_threshold)

# Only tune categories with enough validation positives.
# Otherwise retain the global threshold to reduce overfitting.
for j in range(len(categories)):
    if true[:, j].sum() < 10:
        continue

    best_f1 = -1
    best_t = global_threshold

    for t in np.arange(0.05, 0.81, 0.025):
        pred = (probs[:, j] >= t).astype(int)

        score = f1_score(
            true[:, j],
            pred,
            zero_division=0
        )

        if score > best_f1:
            best_f1 = score
            best_t = float(t)

    thresholds[j] = best_t

pred_global = (probs >= global_threshold).astype(int)
pred_tuned = (probs >= thresholds[None, :]).astype(int)

for name, pred in [
    ("Global threshold", pred_global),
    ("Per-category thresholds", pred_tuned)
]:
    print("\n", name)
    print("Micro-F1:", round(
        f1_score(true, pred, average="micro", zero_division=0), 4
    ))
    print("Macro-F1:", round(
        f1_score(true, pred, average="macro", zero_division=0), 4
    ))
    print("Precision:", round(
        precision_score(true, pred, average="micro", zero_division=0), 4
    ))
    print("Recall:", round(
        recall_score(true, pred, average="micro", zero_division=0), 4
    ))
    print("Predicted positives:", int(pred.sum()))

threshold_df = pd.DataFrame({
    "category": categories,
    "threshold": thresholds,
    "validation_positives": true.sum(axis=0)
})

display(threshold_df)

threshold_df.to_csv(
    "/content/cuad_category_thresholds.csv",
    index=False
)



 Global threshold
Micro-F1: 0.2754
Macro-F1: 0.089
Precision: 0.3636
Recall: 0.2217
Predicted positives: 715

 Per-category thresholds
Micro-F1: 0.1174
Macro-F1: 0.1403
Precision: 0.0666
Recall: 0.497
Predicted positives: 8759


,category,threshold,validation_positives
0,Affiliate License-Licensee,0.050,18
1,Affiliate License-Licensor,0.260,3
2,Agreement Date,0.725,55
3,Anti-Assignment,0.175,67
4,Audit Rights,0.075,71
5,Cap On Liability,0.250,57
6,Change Of Control,0.100,29
7,Competitive Restriction Exception,0.050,10
8,Covenant Not To Sue,0.050,17
9,Document Name,0.475,57


Test predictions

In [10]:

sample_contract = """
This agreement may be terminated by either party
upon thirty days written notice.
"""

result = predict_clauses(
    sample_contract,
    model,
    tokenizer
)

print("Number of outputs:", len(result["probabilities"]))

print("\nPredicted label indices:")
print(np.where(result["predictions"] == 1)[0])

print("\nFirst 10 probabilities:")
print(result["probabilities"][:10])


Number of outputs: 41

Predicted label indices:
[ 0  3  4  7  8 10 14 16 17 19 21 22 25 27 28 32 38 39 40]

First 10 probabilities:
[0.51199037 0.44088367 0.48987353 0.5009758  0.54422444 0.4848301
 0.4908441  0.53987724 0.55657274 0.47886267]


Save the model

In [11]:

SAVE_PATH = "./cuad_distilroberta"

model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print("Model saved successfully!")


Model saved successfully!
